## Question 1. Redpanda version




In [1]:
!docker exec -it 7-streaming_workshop-redpanda-1 rpk version

rpk version: v25.3.9
Git ref:     836b4a36ef6d5121edbb1e68f0f673c2a8a244e2
Build date:  2026 Feb 26 07 48 21 Thu
OS/Arch:     linux/amd64
Go version:  go1.24.3

Redpanda Cluster
  node-1  v25.3.9 - 836b4a36ef6d5121edbb1e68f0f673c2a8a244e2


## Question 2. Sending data to Redpanda

In [24]:
import pandas as pd

In [25]:
url='https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet'

In [26]:
columns = [
    'lpep_pickup_datetime',
    'lpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'tip_amount',
    'total_amount'
]
df = pd.read_parquet(url, columns=columns)

In [27]:
df.head(1)

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.7,1.7,10.0


In [28]:
from dataclasses import dataclass

@dataclass
class Ride:
    lpep_pickup_datetime: str   # Changed from int to str
    lpep_dropoff_datetime: str  # Added
    PULocationID: int
    DOLocationID: int
    passenger_count: int        # Added
    trip_distance: float
    tip_amount: float           # Added
    total_amount: float

In [29]:
def ride_from_row(row):
    return Ride(
        # Convert datetime objects to ISO strings
        lpep_pickup_datetime=row['lpep_pickup_datetime'].isoformat(),
        lpep_dropoff_datetime=row['lpep_dropoff_datetime'].isoformat(),
        PULocationID=int(row['PULocationID']),
        DOLocationID=int(row['DOLocationID']),
        passenger_count=int(row['passenger_count']),
        trip_distance=float(row['trip_distance']),
        tip_amount=float(row['tip_amount']),
        total_amount=float(row['total_amount']),
    )

In [30]:
ride=ride_from_row(df.iloc[0])
ride

Ride(lpep_pickup_datetime='2025-10-01T00:21:47', lpep_dropoff_datetime='2025-10-01T00:24:37', PULocationID=247, DOLocationID=69, passenger_count=1, trip_distance=0.7, tip_amount=1.7, total_amount=10.0)

In [31]:
type(ride.lpep_dropoff_datetime)

str

In [32]:
import json
import dataclasses

def ride_serializer(ride):
    # This still works because the 'ride' object now contains strings
    ride_dict = dataclasses.asdict(ride)
    return json.dumps(ride_dict).encode('utf-8')

In [33]:
from kafka import KafkaProducer

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

### Convert each row to a dictionary and send it to the green-trips topic. You'll need to handle the datetime columns - convert them to strings before serializing to JSON.  

-  Measure the time it takes to send the entire dataset and flush:

In [34]:
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [35]:
df.isnull().sum()

lpep_pickup_datetime        0
lpep_dropoff_datetime       0
PULocationID                0
DOLocationID                0
passenger_count          5015
trip_distance               0
tip_amount                  0
total_amount                0
dtype: int64

In [36]:
df.passenger_count.mean()

np.float64(1.2936870791198396)

In [37]:
# Force the column to be a standard integer, filling holes with 0
df['passenger_count'] = df['passenger_count'].fillna(0).astype(int)

In [38]:
topic_name = 'green-trips'

producer.send(topic_name, value=ride)
producer.flush()

In [39]:
import time

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    #print(f"Sent: {ride}")
    

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

took 36.37 seconds


How long did it take to send the data?

**10 seconds**  
60 seconds  
120 seconds  
300 seconds  

## Question 3. Consumer - trip distance

In [18]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'green-trips'

In [19]:
def ride_deserializer(data):
    json_str = data.decode('utf-8')
    ride_dict = json.loads(json_str)
    return Ride(**ride_dict)

In [20]:
# consumer.subscribe(['green-trips'])
# consumer.seek_to_beginning()

In [21]:
consumer = KafkaConsumer(
    
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='green-trips-class2',
    value_deserializer=ride_deserializer,
    # Stop the loop if no new message arrives for 5 seconds
    consumer_timeout_ms=5000
)

In [22]:
# record=next(consumer)
# record.value

In [23]:
trip_count = 0

# The consumer acts as a 'generator'—it yields messages one by one
for message in consumer:
    ride = message.value # This is your Ride object
    
    if ride.trip_distance > 5.0:
        trip_count += 1
        
    # Optional: Print every 100th trip to see progress
    if trip_count % 100 == 0:
        print(f"Found {trip_count} long trips so far...")

Found 0 long trips so far...
Found 0 long trips so far...
Found 0 long trips so far...
Found 0 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 100 long trips so far...
Found 200 long trips so far...
Found 300 long trips so far...
Found 400 long trips so far...
Found 400 long trips so far...
Found 400 long trips so far...
Found 400 long trips so far...
Found 400 long trips so far...
Found 500 long trips so far...
Found 500 long trips so far...
Found 500 long t

In [ ]:
trip_count

Write a Kafka consumer that reads all messages from the green-trips topic (set auto_offset_reset='earliest').  

Count how many trips have a trip_distance greater than 5.0 kilometers.  

How many trips have trip_distance > 5?  

- 6506
- 7506
- 8506
- **9506**

## Part 2: PyFlink (Questions 4-6)

For the PyFlink questions, you'll adapt the workshop code to work with the green taxi data. The key differences from the workshop:   

Topic name: green-trips (instead of rides)  
Datetime columns use lpep_ prefix (instead of tpep_)  
You'll need to handle timestamps as strings (not epoch milliseconds)  
You can convert string timestamps to Flink timestamps in your source DDL:  

In [ ]:
## src/trip_counter.py

In [ ]:
import os
from pyflink.table import EnvironmentSettings, StreamTableEnvironment

def run_window_job():
    # 1. Setup Environment
    settings = EnvironmentSettings.new_instance().in_streaming_mode().build()
    t_env = StreamTableEnvironment.create(environment_settings=settings)
    
    # CRITICAL: Set parallelism to 1 for Watermark advancement
    t_env.get_config().get_configuration().set_string("parallelism.default", "1")

  # 2. Source DDL (Reading from Redpanda)
    t_env.execute_sql("""
        CREATE TABLE green_trips (
            lpep_pickup_datetime STRING,
            PULocationID INT,
            -- Clean the string and cast it
            event_timestamp AS CAST(REPLACE(lpep_pickup_datetime, 'T', ' ') AS TIMESTAMP(3)),
            WATERMARK FOR event_timestamp AS event_timestamp - INTERVAL '5' SECOND
        ) WITH (
            'connector' = 'kafka',
            'topic' = 'green-trips',
            'properties.bootstrap.servers' = '172.20.0.4:29092',
            'properties.group.id' = 'flink-window-job',
            'scan.startup.mode' = 'earliest-offset',
            'format' = 'json'
        );
    """)

 
    # 3. Sink DDL (Writing to PostgreSQL)
    t_env.execute_sql("""
        CREATE TABLE sink_postgres (
            window_start TIMESTAMP(3),
            PULocationID INT,
            num_trips BIGINT
        ) WITH (
            'connector' = 'jdbc',
            'url' = 'jdbc:postgresql://postgres:5432/postgres', 
            'table-name' = 'processed_trips_window',
            'username' = 'postgres',   -- CHANGED FROM 'your_user'
            'password' = 'postgres',   -- CHANGED FROM 'your_password'
            'driver' = 'org.postgresql.Driver'
        );
    """)

    # 4. The Transformation (Tumbling Window)
    # Using Table API or SQL to count trips in 5-minute blocks
    t_env.execute_sql("""
        INSERT INTO sink_postgres
        SELECT 
            TUMBLE_START(event_timestamp, INTERVAL '5' MINUTES) AS window_start,
            PULocationID,
            COUNT(*) as num_trips
        FROM green_trips
        GROUP BY 
            TUMBLE(event_timestamp, INTERVAL '5' MINUTES),
            PULocationID
    """)

if __name__ == '__main__':
    run_window_job()

## copy the file to docker  
```bash
docker cp src/job/trip_counter.py 7-streaming_workshop-jobmanager-1:/opt/src/job/trip_counter.py

```

In [ ]:
## docker exec -it 7-streaming_workshop-jobmanager-1 flink run -py /opt/src/job/trip_counter.py

SELECT PULocationID, num_trips  
FROM processed_trips_window  
ORDER BY num_trips DESC  
LIMIT 3;  

## Question 4. Tumbling window - pickup location
Which PULocationID had the most trips in a single 5-minute window?  


74  


In [ ]:
## session_job.py

In [ ]:
import os
from pyflink.table import EnvironmentSettings, TableEnvironment

# 1. Initialize the Environment
env_settings = EnvironmentSettings.new_instance().in_streaming_mode().build()
t_env = TableEnvironment.create(env_settings)

# CRITICAL: This allows the watermark to move even if some Kafka partitions are empty
t_env.get_config().get_configuration().set_string("table.exec.source.idle-timeout", "10s")

# 2. Source DDL (Handles the 'T' in your strings)
t_env.execute_sql("""
    CREATE TABLE green_trips (
        lpep_pickup_datetime STRING,
        PULocationID INT,
        event_timestamp AS CAST(REPLACE(lpep_pickup_datetime, 'T', ' ') AS TIMESTAMP(3)),
        WATERMARK FOR event_timestamp AS event_timestamp - INTERVAL '5' SECOND
    ) WITH (
        'connector' = 'kafka',
        'topic' = 'green-trips',
        'properties.bootstrap.servers' = '172.20.0.4:29092',
        'properties.group.id' = 'flink-session-group-v1',
        'scan.startup.mode' = 'earliest-offset',
        'format' = 'json'
    );
""")

# 3. Sink DDL (Connects to Postgres)
t_env.execute_sql("""
    CREATE TABLE session_trips_sink (
        window_start TIMESTAMP(3),
        window_end TIMESTAMP(3),
        pulocationid INT,
        num_trips BIGINT
    ) WITH (
        'connector' = 'jdbc',
        'url' = 'jdbc:postgresql://postgres:5432/postgres',
        'table-name' = 'session_trips',
        'username' = 'postgres',
        'password' = 'postgres'
    );
""")

# 4. The Session Window Query
# This groups trips that occur within 5 minutes of each other.
t_env.execute_sql("""
    INSERT INTO session_trips_sink
    SELECT 
        SESSION_START(event_timestamp, INTERVAL '5' MINUTE) AS window_start,
        SESSION_END(event_timestamp, INTERVAL '5' MINUTE) AS window_end,
        PULocationID,
        COUNT(*) AS num_trips
    FROM green_trips
    GROUP BY SESSION(event_timestamp, INTERVAL '5' MINUTE), PULocationID
""")

## Question 5. Session window - longest streak

Create another Flink job that uses a session window with a 5-minute gap on PULocationID, using lpep_pickup_datetime as the event time with a 5-second watermark tolerance.

A session window groups events that arrive within 5 minutes of each other. When there's a gap of more than 5 minutes, the window closes.

Write the results to a PostgreSQL table and find the PULocationID with the longest session (most trips in a single session).

How many trips were in the longest session?


**81**

In [ ]:
## docker cp src/session_job.py 7-streaming_workshop-jobmanager-1:/opt/src/session_job.py

In [40]:
# Run the job
## docker exec -it 7-streaming_workshop-jobmanager-1 flink run -py /src/session_job.py

## docker exec -it 7-streaming_workshop-postgres-1 psql -U postgres -d postgres -c   
"SELECT num_trips  
FROM session_trips  
ORDER BY num_trips  
DESC LIMIT 1;"  

In [ ]:
Question 6. Tumbling window - largest tip
Create a Flink job that uses a 1-hour tumbling window to compute the total tip_amount per hour (across all locations).

Which hour had the highest total tip amount?

2025-10-01 18:00:00
2025-10-16 18:00:00
2025-10-22 08:00:00
2025-10-30 16:00:00